# Extract Initial Data
This notebook walks through the steps required to extract the initial dataset on the UKB-RAP.
In order to run this, access to the UK Biobank is necessary, as well as a running python-spark environment on the UKB RAP

The resulting dataset contains the data fields of interest for all samples.  
**Note:**
The notebooks used to create the sample-QCED datasets are based on or inspired by the GWAS AD Guide for the UK Biobank (
[GWAS_AD_Tutorial](https://dnanexus.gitbook.io/uk-biobank-rap/science-corner/gwas-ex)).
This guide focuses on creating an AD-By-Proxy dataset and to perform early sample QC-steps.
The second part of the original tutorial, which covers GWAS settings and performing GWAS, is not included here.

## Retrieve Project Data 
To get started, we need to find out our project ID. 
In case this is not known, it can be done using this code here from the tutorial: 


In [ ]:
# Import packages 
import dxpy
import subprocess

# Automatically discover dispensed dataset ID and load the dataset 
dispensed_dataset_id = dxpy.find_one_data_object(typename='Dataset', name='app*.dataset', folder='/', name_mode='glob')['id']

# Get project ID
project_id = dxpy.find_one_project()["id"]
dataset = (':').join([project_id, dispensed_dataset_id])

cmd = ["dx", "extract_dataset", dataset, "-ddd", "--delimiter", ","]
subprocess.check_call(cmd)

After this succeeds, replace `<project:id>` in the following code with the respective infos. 

In [ ]:
import subprocess
dataset = '<project:id>.dataset'
cmd = ["dx", "extract_dataset", dataset, "-ddd", "--delimiter", ","]
subprocess.check_call(cmd)

This creates the three files:
```bash
$ls *.csv
id.dataset.codings.csv
id.dataset.data_dictionary.csv
id.dataset.entity_dictionary.csv
```
We start by loading the participants and their metadata: 

In [ ]:
import pandas as pd
data_dict_csv='id.dataset.data_dictionary.csv'
data_dict_df = pd.read_csv(data_dict_csv, delimiter=",")
data_dict_df.head()

In the next step, we select the fields we are interested in for our study. 
This can be done individually, please refer to the AMS Showcase and search for the field ids. 

In [ ]:
from distutils.version import LooseVersion

def field_names_for_ids(field_id):
	field_names = ["eid"]
	for _id in field_id:
		select_field_names = list(data_dict_df[data_dict_df.name.str.match(r'^p{}(_i\d+)?(_a\d+)?$'.format(_id))].name.values)
		field_names += select_field_names
	field_names = sorted([field for field in field_names], key=lambda n: LooseVersion(n))
		
	field_names = [f"participant.{f}" for f in field_names]
	return ",".join(field_names)

field_ids = ['31', '21022', '22001', '22006', '22009', '22019', '22021', '22027',
				 '41270', '20107', '2946', '1807',
				 '20110', '3526', '1845', '21000']
# initially, see post
field_ids = ['31', '21022', '22001', '22006', '22009', '22019', '22021', '22027',
				 '41270', '20107', '2946', '1807',
				 '20110', '3526', '1845', '21000']
# added for wgs data
# 23080,Sequencing Provider
# 23081,Sample plate ID
# 23082,Shipment batch number
# 23083,Sample quant reading (UKB)
# 23084,Sample quant reading (sequence provider)
# 23085,Library prep plate barcode
# 23086,Library prep plate position (well)
# 23087,Yield
# 23088,Proportion of mapped read pairs
# 23089,Coverage
# 23090,Average batch coverage
# 23091,Freemix verify BAM ID
# 23092,NRD genotyping
# 23093,Quality requirements achieved
# 23094,Read haps
wgs_ids=[23080, 23081, 23082, 23083, 23084, 23085, 23086, 23087, 23088, 23089, 23090, 23091, 23092, 23093, 23094]
field_ids.append(wgs_ids)
# 33 - Date of birth
# 131036 - Date G30 first reported (alzheimer's disease)
# 130836 - Date F00 first reported (dementia in alzheimer's disease)
report_ids=[33, 131036, 30836 ]
field_ids.append(report_ids)
field_names = field_names_for_ids(field_ids)

## Extract SQL

In [ ]:
# Extract dataset with selected fields and save as SQL-file
outfile_name = 'extracted_data.sql'
dataset = '<project:id>.dataset'
cmd = ["dx", "extract_dataset", dataset, "--fields", field_names, "--delimiter", ",", "--output", outfile_name, "--sql"]
subprocess.check_call(cmd)

In [ ]:
# upload the SQL file, to be on the safe side
filename = 'extracted_data.sql'
cmd = ["dx", "upload", filename]
subprocess.check_call(cmd)

## Convert to Dataframe

In [ ]:
import pyspark

sc = pyspark.SparkContext()
spark = pyspark.sql.SparkSession(sc)

with open("extracted_data.sql", "r") as file:
	retrieve_sql=""
	for line in file: 
		retrieve_sql += line.strip()  
				 
temp_df = spark.sql(retrieve_sql.strip(";"))
pdf = temp_df.toPandas()
pdf.head()

In [ ]:
pdf.to_csv("df_from_sql.csv", sep='\t', na_rep='NA', index=False, quoting=3)

In [ ]:
cmd = ["dx", "upload", "df_from_sql.csv"]
subprocess.check_call(cmd)